# Rogers × Databricks — Combined Cell Traffic EDA (Waterfront · UBC · Park Royal)

**Goal:** compare a downtown transit hub, a university campus and a shopping mall: who arrives, when, how long they stay, and when it's unusual. This is the groundwork for the smart-community **transit + security** solution.

**Data:** three tables with the same schema: `location_name, longitude, latitude, timestamp, origin, dwell_time`.
* `timestamp`: when a device attached to the site's cell tower
* `dwell_time`: **minutes** attached to the tower (the tower covers more than the named place)
* `origin`: the device's **home location**
* There is no device ID, so counts are *device connections*, not unique people

Heavy aggregation stays in Spark, and only small aggregates go to pandas. Runs on serverless (no `.cache()`).
Everything time-of-day runs at **15-minute** resolution (the single-site notebook fell back to daily buckets for occupancy and surges).

| # | Section | Question |
|---|---|---|
| 0 | Setup | config, union of the three sites, `silver_visits` |
| 1 | Data quality | nulls, duplicates, ranges, invalid dwell, coverage (per site) |
| 2 | Synthetic-data checks | generator artefacts, burstiness |
| 3 | Origin volume | catchment of each site; residents vs visitors |
| 4 | Arrival patterns over time | daily/monthly volume, weekday × hour rhythm, holidays, peak days |
| 5 | Origin × time | when each home area arrives; commuter signatures |
| 6 | Dwell time | how long people stay, by site, origin and hour |
| 7 | Occupancy (15-min) | devices present at once, and who they are |
| 8 | Security signals | surges, origin surges, crowd-mix shift, rapid build-up, long / overnight stays |
| 9 | Relationships | crowding vs dwell, cross-site links, similar origins |
| 10 | Catchment map | flows from home areas to each site, travel distance |
| 11 | KPI tables | gold tables for the dashboard / forecasting |

In [0]:
%pip install -q plotly seaborn

## 0. Setup

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Window
from builtins import min, max, sum, abs, round   # restores Python built-ins if `from pyspark.sql.functions import *` ran anywhere
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.graph_objects as go

In [0]:
# ---------------- CONFIG ----------------
SITES = {                                     # site label -> source table
    "Waterfront": "rogersdatabricks.default.synthetic_data_waterfront",
    "UBC":        "rogersdatabricks.default.synthetic_data_ubc",
    "Park Royal": "rogersdatabricks.default.synthetic_park_royal_mall",
}
SILVER_SCHEMA = "rogersdatabricks.silver"     # cleaned record-level data (silver_visits)
GOLD_SCHEMA = "rogersdatabricks.gold"         # aggregated tables for the dashboard / forecasting
LOCAL_TZ = "UTC"                              # "UTC" = analyse clock times exactly as written (Waterfront peaks at 08/16 suggest they're already local)
DROP_DUPLICATES = True
RES_MIN = 15                                  # slot size for flow / occupancy / anomalies
TOP_N = 12                                    # origins shown individually; the rest become "Other"
MAX_DWELL_MIN = 1440                          # dwell above 24h is treated as invalid
WARMUP_DAYS = 1                               # occupancy is under-counted until one max dwell has passed
BASELINE_WEEKS = 8                            # trailing same-weekday / same-slot weeks in each baseline
Z_THRESH = 3.5                                # robust z-score threshold
MIN_ARRIVALS = 20                             # ignore surges in near-empty slots
MIN_MIX = 200                                 # min arrivals in an hour before scoring its origin mix
DWELL_BINS = [0, 15, 180, 600, MAX_DWELL_MIN, float("inf")]
DWELL_LABELS = ["passing (<15m)", "visiting (15m–3h)", "staying (3–10h)", "very long (10–24h)", "invalid (>24h)"]
ORIGIN_TIER = {                               # origins outside Metro Vancouver; everything else = "Metro Vancouver"
    "Ontario": "Out of province", "Alberta": "Out of province", "Manitoba": "Out of province",
    "Atlantic Canada": "Out of province", "Saskatchewan and Territories": "Out of province",
    "British Columbia Other": "Rest of BC", "International": "International",
}
BC_HOLIDAYS = {                               # BC statutory holidays in the data window (verify against the official calendar)
    "2025-11-11": "Remembrance Day", "2025-12-25": "Christmas Day", "2026-01-01": "New Year's Day",
    "2026-02-16": "Family Day", "2026-04-03": "Good Friday", "2026-05-18": "Victoria Day",
    "2026-07-01": "Canada Day", "2026-08-03": "BC Day",
}
CAPACITY_SCENARIOS = {}                       # e.g. {"Waterfront": [3000], "Park Royal": [5000]}: reference lines on occupancy charts
# ----------------------------------------

RES_S = RES_MIN * 60
SITE_ORDER = list(SITES)
SITE_COLORS = dict(zip(SITE_ORDER, ["#1f77b4", "#2ca02c", "#d62728"]))
TIER_ORDER = ["Metro Vancouver", "Rest of BC", "Out of province", "International"]
DOW = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

spark.conf.set("spark.sql.session.timeZone", LOCAL_TZ)

plt.rcParams.update({
    "figure.figsize": (13, 5), "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 13,
})
sns.set_palette("tab10")

def show():
    plt.tight_layout()
    plt.show()

def materialize(sdf, name):
    """Serverless-safe replacement for .cache(): write to Delta and read back."""
    full_name = f"{SILVER_SCHEMA}.{name}"
    sdf.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_name)
    return spark.table(full_name)

def save_pdf(pdf, name):
    """Write a pandas frame to Delta (object columns: NaN -> None so Spark infers a clean type)."""
    pdf = pdf.copy()
    for c in pdf.select_dtypes("object").columns:
        pdf[c] = pdf[c].astype(object).where(pdf[c].notna(), None)
    spark.createDataFrame(pdf).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{GOLD_SCHEMA}.{name}")

def slot(col):
    return F.expr(f"timestamp_seconds(floor(unix_timestamp({col}) / {RES_S}) * {RES_S})")

def fmt_k(x, _=None):
    return f"{x/1e6:.1f}M" if abs(x) >= 1e6 else (f"{x/1e3:.0f}K" if abs(x) >= 1e3 else f"{x:.0f}")

def site_axes(nrows=1, ncols=3, w=6.0, h=4.5, **kw):
    fig, axes = plt.subplots(nrows, ncols, figsize=(w * ncols, h * nrows), squeeze=False, **kw)
    return fig, axes

def nanmad(x):
    x = x[~np.isnan(x)]
    return np.median(np.abs(x - np.median(x))) if len(x) else np.nan

def trailing_z(frame, col, keys, weeks=BASELINE_WEEKS, exclude=None):
    """Robust z of `col` against the same `keys` group over its previous `weeks` occurrences.
    `frame` must be sorted by time within each group. Uses past values only, so it also works for backtests / live data.
    Scale = max(1.4826·MAD, √baseline, 1) so quiet slots don't give huge z-scores."""
    hist = frame[col] if exclude is None else frame[col].where(~exclude)
    g = hist.groupby([frame[k] for k in keys])
    roll = lambda s: s.shift(1).rolling(weeks, min_periods=max(2, weeks // 2))
    base = g.transform(lambda s: roll(s).median())
    mad = g.transform(lambda s: roll(s).apply(nanmad, raw=True))
    scale = np.fmax(np.fmax(1.4826 * mad, np.sqrt(base.clip(lower=0))), 1.0)
    return base, (frame[col] - base) / scale

In [0]:
# Load each site, drop exact duplicates, union with a `site` label
counts, parts = [], []
for site, table in SITES.items():
    t = spark.table(table)
    n_raw = t.count()
    if DROP_DUPLICATES:
        t = t.dropDuplicates()
    n_used = t.count()
    counts.append({"site": site, "table": table, "raw_rows": n_raw, "duplicates": n_raw - n_used,
                   "dup_pct": (n_raw - n_used) / n_raw, "rows_used": n_used})
    parts.append(t.withColumn("site", F.lit(site)))
counts = pd.DataFrame(counts)
display(counts)

union = parts[0]
for p in parts[1:]:
    union = union.unionByName(p)

seg = F.when(F.col("dwell_time") < DWELL_BINS[1], DWELL_LABELS[0])
for hi, lab in zip(DWELL_BINS[2:-1], DWELL_LABELS[1:-1]):
    seg = seg.when(F.col("dwell_time") <= hi, lab)
seg = seg.otherwise(DWELL_LABELS[-1])

ts_expr = F.to_timestamp("timestamp") if dict(union.dtypes)["timestamp"] == "string" else F.col("timestamp")
tier_map = F.create_map(*[F.lit(x) for kv in ORIGIN_TIER.items() for x in kv])
holiday_map = F.create_map(*[F.lit(x) for kv in BC_HOLIDAYS.items() for x in kv])

enriched = (union
    .withColumn("ts", ts_expr)
    .withColumn("dwell_time", F.col("dwell_time").cast("double"))
    .withColumn("dwell_valid", F.col("dwell_time") <= MAX_DWELL_MIN)
    .withColumn("end_ts", F.expr("timestamp_seconds(unix_timestamp(ts) + CAST(dwell_time * 60 AS BIGINT))"))
    .withColumn("slot", slot("ts"))
    .withColumn("end_slot", slot("end_ts"))
    .withColumn("date", F.to_date("ts"))
    .withColumn("month", F.date_format("ts", "yyyy-MM"))
    .withColumn("hour", F.hour("ts"))
    .withColumn("minute", F.minute("ts"))
    .withColumn("second", F.second("ts"))
    .withColumn("dow", F.expr("weekday(ts)"))          # 0 = Monday
    .withColumn("is_weekend", F.col("dow") >= 5)
    .withColumn("holiday", holiday_map[F.date_format("ts", "yyyy-MM-dd")])
    .withColumn("tier", F.coalesce(tier_map[F.col("origin")], F.lit("Metro Vancouver")))
    .withColumn("visitor_type", F.when(F.col("tier") == "Metro Vancouver", "resident").otherwise("visitor"))
    .withColumn("dwell_segment", seg))

df = materialize(enriched, "silver_visits")
dv = df.filter("dwell_valid")                 # valid-dwell records: used for dwell stats and occupancy
print(f"silver_visits: {df.count():,} rows")

In [0]:
# Time span per site: do the three sites cover the same window?
span = (df.groupBy("site")
          .agg(F.count("*").alias("rows"), F.min("ts").alias("first"), F.max("ts").alias("last"),
               F.max("end_ts").alias("last_end"), F.countDistinct("date").alias("n_days"),
               F.countDistinct("origin").alias("n_origins"))
          .toPandas().set_index("site").loc[SITE_ORDER])
span["rows_per_day"] = span["rows"] / span["n_days"]
display(span.reset_index())

if span["first"].dt.date.nunique() > 1 or span["last"].dt.date.nunique() > 1:
    print(f"⚠ Sites cover different windows. Overlap: {span['first'].max()} → {span['last'].min()}. Compare volumes on the overlap only.")
else:
    print("All sites cover the same window.")

SITE_START = {s: pd.Timestamp(span.loc[s, "first"]).floor(f"{RES_MIN}min") for s in SITE_ORDER}
SITE_END = {s: pd.Timestamp(span.loc[s, "last"]).floor(f"{RES_MIN}min") for s in SITE_ORDER}

## 1. Data quality

In [0]:
# Schemas should match across the three source tables
schemas = {s: dict(spark.table(t).dtypes) for s, t in SITES.items()}
display(pd.DataFrame(schemas).reset_index().rename(columns={"index": "column"}))
display(df.limit(20))

In [0]:
base_cols = ["location_name", "longitude", "latitude", "timestamp", "origin", "dwell_time"]
nulls = df.groupBy("site").agg(*[F.sum(F.col(c).isNull().cast("int")).alias(c) for c in base_cols]).toPandas()
display(nulls.set_index("site").loc[SITE_ORDER].reset_index())

# Location / coordinate consistency: expect exactly one row per site
display(df.groupBy("site", "location_name", "latitude", "longitude").count().orderBy("site"))

In [0]:
# Dwell summary per site (all records, including invalid)
qs = [0.01, 0.05, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 0.999]
dsum = (df.groupBy("site")
          .agg(F.expr(f"percentile_approx(dwell_time, array({', '.join(map(str, qs))}), 10000)").alias("q"),
               F.mean("dwell_time").alias("mean"), F.stddev("dwell_time").alias("std"),
               F.min("dwell_time").alias("min"), F.max("dwell_time").alias("max"),
               F.sum((F.col("dwell_time") <= 0).cast("int")).alias("non_positive"),
               F.sum((F.col("dwell_time") != F.floor("dwell_time")).cast("int")).alias("non_integer"),
               F.sum((~F.col("dwell_valid")).cast("int")).alias("over_24h"),
               F.count("*").alias("rows"))
          .toPandas().set_index("site").loc[SITE_ORDER])
dsum[[f"p{q * 100:g}" for q in qs]] = pd.DataFrame(dsum["q"].tolist(), index=dsum.index)
dsum["pct_over_24h"] = dsum["over_24h"] / dsum["rows"]
display(dsum.drop(columns="q").reset_index())

In [0]:
# Invalid dwell (> 24h): which values, and which home areas? (checks whether they explain out-of-province long stays)
display(df.filter(~F.col("dwell_valid")).groupBy("site", "dwell_time").count()
          .orderBy(F.desc("count")).limit(20))

inv = (df.groupBy("site", "origin")
         .agg(F.avg((~F.col("dwell_valid")).cast("int")).alias("share_over_24h"))
         .toPandas().pivot(index="origin", columns="site", values="share_over_24h").reindex(columns=SITE_ORDER))
inv = inv.loc[inv.max(axis=1).sort_values(ascending=False).index]
plt.figure(figsize=(8, max(5, 0.32 * len(inv))))
sns.heatmap(inv, cmap="Reds", annot=True, fmt=".2%", cbar_kws={"label": "share of records > 24h"})
plt.title("Invalid dwell (> 24h) by home area and site")
show()

In [0]:
# Origin labels per site: typos, casing, whitespace, or home areas missing from a site
org_site = (df.groupBy("origin", "site").count().toPandas()
              .pivot(index="origin", columns="site", values="count").reindex(columns=SITE_ORDER))
display(org_site.reset_index())
print("Home areas missing from at least one site:", org_site[org_site.isna().any(axis=1)].index.tolist())
print("Home areas treated as Metro Vancouver residents (check none are really out-of-region):")
print(sorted(set(org_site.index) - set(ORIGIN_TIER)))

In [0]:
# Coverage: missing days, and share of 15-min slots with no arrivals, by hour of day
cov_days = df.groupBy("site", "date").count().toPandas()
for s in SITE_ORDER:
    d = pd.to_datetime(cov_days.loc[cov_days["site"] == s, "date"])
    missing = pd.date_range(d.min(), d.max(), freq="D").difference(d)
    print(f"{s}: {len(d)} days with data, {len(missing)} missing" + (f" (first: {list(missing[:5])})" if len(missing) else ""))

cs = df.groupBy("site", "slot").count().toPandas()
rows = []
for s in SITE_ORDER:
    idx = pd.date_range(SITE_START[s], SITE_END[s], freq=f"{RES_MIN}min")
    present = pd.Series(1, index=pd.DatetimeIndex(cs.loc[cs["site"] == s, "slot"])).reindex(idx, fill_value=0)
    rows.append(pd.DataFrame({"site": s, "hour": idx.hour, "empty": present.values == 0}))
empty = pd.concat(rows).groupby(["site", "hour"])["empty"].mean().unstack("site")[SITE_ORDER]
ax = empty.plot(figsize=(13, 4), marker="o", color=[SITE_COLORS[s] for s in SITE_ORDER])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title(f"Share of {RES_MIN}-min slots with zero arrivals, by hour of day")
show()

## 2. Synthetic-data checks
Generated data often has tell-tale patterns (only some minutes used, uniform seconds, perfectly smooth rates). Knowing them stops us from presenting generator artefacts as insights.

In [0]:
hms = {c: df.groupBy("site", c).count().toPandas() for c in ["hour", "minute", "second"]}
fig, axes = site_axes(3, 3, w=6, h=3.3)
for i, s in enumerate(SITE_ORDER):
    for j, c in enumerate(["hour", "minute", "second"]):
        p = hms[c][hms[c]["site"] == s].sort_values(c)
        ax = axes[i][j]
        ax.bar(p[c], p["count"], color=SITE_COLORS[s])
        ax.set_title(f"{s}: records by {c}")
        ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
show()

mm = hms["minute"]
for s in SITE_ORDER:
    p = mm[mm["site"] == s]
    print(f"{s}: records in :30–:59 / :00–:29 = {p.loc[p['minute'] >= 30, 'count'].sum() / p.loc[p['minute'] < 30, 'count'].sum():.3f}  (1.000 = no artefact)")

In [0]:
# Burstiness: arrivals per second vs a random (Poisson) process
per_sec = df.groupBy("site", F.date_trunc("second", "ts").alias("s")).count()
dist = per_sec.groupBy("site", "count").agg(F.count("*").alias("n_seconds")).toPandas()
rows = []
for s in SITE_ORDER:
    d = dist.loc[dist["site"] == s, ["count", "n_seconds"]]
    total_secs = int((span.loc[s, "last"] - span.loc[s, "first"]).total_seconds()) + 1
    zero_secs = total_secs - int(d["n_seconds"].sum())
    if zero_secs > 0:
        d = pd.concat([d, pd.DataFrame({"count": [0], "n_seconds": [zero_secs]})])
    mean = np.average(d["count"], weights=d["n_seconds"])
    var = np.average((d["count"] - mean) ** 2, weights=d["n_seconds"])
    rows.append({"site": s, "mean_per_sec": mean, "variance": var, "index_of_dispersion": var / mean})
display(pd.DataFrame(rows))
print("≈1 → random, independent arrivals · >1 → bursty (groups arriving together) · <1 → unnaturally regular")

## 3. Origin volume: which home areas feed each site?

In [0]:
org_vol = df.groupBy("site", "origin", "tier").count().toPandas()
org_vol["share"] = org_vol["count"] / org_vol.groupby("site")["count"].transform("sum")
org_vol = org_vol.sort_values(["site", "count"], ascending=[True, False])
TOP_ORIGINS = {s: org_vol.loc[org_vol["site"] == s, "origin"].head(TOP_N).tolist() for s in SITE_ORDER}
ORIGIN_ORDER = org_vol.groupby("origin")["count"].sum().sort_values(ascending=False).index.tolist()

n_show = min(25, org_vol.groupby("site").size().max())
fig, axes = site_axes(1, 3, w=6.5, h=max(5, 0.32 * n_show))
for ax, s in zip(axes[0], SITE_ORDER):
    p = org_vol[org_vol["site"] == s].head(n_show).iloc[::-1]
    bars = ax.barh(p["origin"], p["count"], color=SITE_COLORS[s])
    for b, sh in zip(bars, p["share"]):
        ax.text(b.get_width(), b.get_y() + b.get_height() / 2, f" {sh:.1%}", va="center", fontsize=8)
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
    ax.set_title(f"{s}: top {n_show} home areas")
show()

In [0]:
# Catchment comparison: share of each site's records by home area
cpiv = org_vol.pivot(index="origin", columns="site", values="share").reindex(index=ORIGIN_ORDER, columns=SITE_ORDER).fillna(0)
plt.figure(figsize=(9, max(6, 0.32 * len(cpiv))))
sns.heatmap(cpiv, cmap="mako_r", annot=True, fmt=".1%", cbar_kws={"label": "share of site's records"})
plt.title("Catchment: share of each site's records by home area")
show()

# How different are the catchments? Jensen–Shannon divergence between each pair of sites' home-area mix.
# ≈0 means near-identical mixes (e.g. a generator that ignores site); ~0.05+ is a clearly different catchment.
def jsd(p, q):
    m = 0.5 * (p + q)
    with np.errstate(divide="ignore", invalid="ignore"):
        return 0.5 * np.nansum(np.where(p > 0, p * np.log(p / m), 0)) + 0.5 * np.nansum(np.where(q > 0, q * np.log(q / m), 0))
pairs = [(a, b, jsd(cpiv[a].values, cpiv[b].values)) for i, a in enumerate(SITE_ORDER) for b in SITE_ORDER[i + 1:]]
display(pd.DataFrame(pairs, columns=["site_a", "site_b", "catchment_jsd"]))

In [0]:
# Concentration: how many home areas supply 80% of each site's traffic?
fig, ax = plt.subplots(figsize=(10, 5))
for s in SITE_ORDER:
    cum = org_vol.loc[org_vol["site"] == s, "share"].cumsum().values
    n80 = int(np.argmax(cum >= 0.8)) + 1
    ax.plot(np.arange(1, len(cum) + 1), cum, marker="o", ms=3, color=SITE_COLORS[s], label=f"{s}: {n80} of {len(cum)} → 80%")
ax.axhline(0.8, ls="--", color="grey")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel("number of home areas (ranked by volume)")
ax.set_ylabel("cumulative share of records")
ax.set_title("Catchment concentration")
ax.legend()
show()

In [0]:
# Residents vs visitors by tier
tiers = org_vol.groupby(["site", "tier"])["count"].sum().unstack().reindex(index=SITE_ORDER, columns=TIER_ORDER).fillna(0)
tiers = tiers.div(tiers.sum(axis=1), axis=0)
ax = tiers.plot(kind="barh", stacked=True, figsize=(12, 3.5), color=["#4C78A8", "#72B7B2", "#F58518", "#E45756"])
for i, s in enumerate(SITE_ORDER):
    ax.text(1.01, i, f"visitors {1 - tiers.loc[s, 'Metro Vancouver']:.1%}", va="center")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(bbox_to_anchor=(1.15, 1), loc="upper left")
ax.set_title("Home-area tier by site")
show()

## 4. Arrival patterns over time

In [0]:
daily = df.groupBy("site", "date").count().toPandas()
daily["date"] = pd.to_datetime(daily["date"])
daily = daily.sort_values("date")
daily["dow"] = daily["date"].dt.dayofweek
daily["same_dow_median"] = daily.groupby(["site", "dow"])["count"].transform(lambda s: s.shift(1).rolling(4, min_periods=2).median())
daily["vs_normal"] = daily["count"] / daily["same_dow_median"]
dpiv = daily.pivot(index="date", columns="site", values="count")[SITE_ORDER]

fig, axes = plt.subplots(2, 1, figsize=(16, 9), sharex=True)
for s in SITE_ORDER:
    axes[0].plot(dpiv.index, dpiv[s], label=s, color=SITE_COLORS[s], lw=1.2)
    axes[1].plot(dpiv.index, dpiv[s] / dpiv[s].median(), label=s, color=SITE_COLORS[s], lw=1.2)
for d in pd.to_datetime(list(BC_HOLIDAYS)):
    for ax in axes:
        ax.axvline(d, color="grey", ls=":", lw=1)
axes[0].set_title("Daily records (dotted = BC stat holiday)")
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
axes[1].set_title("Daily records relative to each site's median day")
axes[1].axhline(1, color="black", lw=0.8)
axes[0].legend()
show()

In [0]:
# Monthly level, weekday vs weekend (average records per day, indexed to each site's mean)
mo = df.groupBy("site", "month", "is_weekend").agg(F.count("*").alias("n"), F.countDistinct("date").alias("days")).toPandas()
mo["per_day"] = mo["n"] / mo["days"]
fig, axes = site_axes(1, 3, w=6, h=4.5, sharey=True)
for ax, s in zip(axes[0], SITE_ORDER):
    p = mo[mo["site"] == s].pivot(index="month", columns="is_weekend", values="per_day").sort_index()
    p = p / np.nanmean(p.values)
    ax.plot(p.index, p[False], marker="o", color=SITE_COLORS[s], label="weekday")
    ax.plot(p.index, p[True], marker="s", ls="--", color=SITE_COLORS[s], alpha=0.6, label="weekend")
    ax.axhline(1, color="grey", lw=0.8)
    ax.set_title(f"{s}: records per day by month (indexed)")
    ax.tick_params(axis="x", rotation=60)
    ax.legend()
show()

In [0]:
# Weekday × hour: average records per hour (holidays excluded)
dh = (df.filter(F.col("holiday").isNull())
        .groupBy("site", "date", "dow", "hour").count()
        .groupBy("site", "dow", "hour").agg(F.avg("count").alias("avg")).toPandas())
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
for ax, s in zip(axes, SITE_ORDER):
    piv = dh[dh["site"] == s].pivot(index="dow", columns="hour", values="avg").sort_index()
    piv.index = [DOW[i] for i in piv.index]
    sns.heatmap(piv, cmap="rocket_r", ax=ax, cbar_kws={"label": "avg records / hour"})
    ax.set_title(f"{s}: average records, weekday × hour")
show()

In [0]:
# Daily shape, normalised so sites of different size compare directly
prof = (dh.assign(day_type=np.where(dh["dow"] >= 5, "weekend", "weekday"))
          .groupby(["site", "day_type", "hour"])["avg"].mean().reset_index())
prof["share"] = prof["avg"] / prof.groupby(["site", "day_type"])["avg"].transform("sum")
fig, axes = site_axes(1, 2, w=9, h=5, sharey=True)
for ax, dt in zip(axes[0], ["weekday", "weekend"]):
    for s in SITE_ORDER:
        p = prof[(prof["site"] == s) & (prof["day_type"] == dt)]
        ax.plot(p["hour"], p["share"], marker="o", label=s, color=SITE_COLORS[s])
    ax.set_title(f"{dt.title()} arrival profile (share of the day's arrivals)")
    ax.set_xticks(range(24))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend()
show()

# By what hour has a typical weekday reached 25 / 50 / 75 / 90% of its arrivals?
wk = prof[prof["day_type"] == "weekday"].sort_values("hour").copy()
wk["cum"] = wk.groupby("site")["share"].cumsum()
marks = pd.DataFrame({f"{q:.0%} by": wk[wk["cum"] >= q].groupby("site")["hour"].min().map(lambda h: f"{h:02d}:59")
                      for q in [0.25, 0.5, 0.75, 0.9]}).loc[SITE_ORDER]
display(marks.reset_index())

In [0]:
# Holidays vs a normal day of the same weekday
hol = daily[daily["date"].dt.strftime("%Y-%m-%d").isin(BC_HOLIDAYS)].copy()
hol["holiday"] = hol["date"].dt.strftime("%Y-%m-%d").map(BC_HOLIDAYS)
display(hol.pivot(index=["date", "holiday"], columns="site", values="vs_normal").reindex(columns=SITE_ORDER).round(2).reset_index())

# Peak days per site: highest volume relative to the same weekday's trailing median
peaks = daily.dropna(subset=["vs_normal"]).sort_values("vs_normal", ascending=False).groupby("site").head(10)
peaks["holiday"] = peaks["date"].dt.strftime("%Y-%m-%d").map(BC_HOLIDAYS)
display(peaks.sort_values(["site", "vs_normal"], ascending=[True, False])[["site", "date", "count", "same_dow_median", "vs_normal", "holiday"]])

## 5. Origin × time: when does each home area arrive?

In [0]:
# Weekday (non-holiday) arrival profile of each site's top 20 home areas, row-normalised
oh = (df.filter(~F.col("is_weekend") & F.col("holiday").isNull())
        .groupBy("site", "origin", "hour").count().toPandas())
fig, axes = plt.subplots(1, 3, figsize=(27, 8))
for ax, s in zip(axes, SITE_ORDER):
    top = org_vol.loc[org_vol["site"] == s, "origin"].head(20)
    p = oh[oh["site"] == s].pivot(index="origin", columns="hour", values="count").reindex(top).fillna(0)
    p = p.div(p.sum(axis=1), axis=0)
    sns.heatmap(p, cmap="mako_r", vmin=0, vmax=0.15, ax=ax, cbar=(s == SITE_ORDER[-1]),
                cbar_kws={"label": "share of origin's weekday arrivals"})
    ax.set_title(f"{s}: when each home area arrives (weekdays)")
    ax.set_ylabel("")
show()

In [0]:
# Commuter signature: share of weekday arrivals in the morning (06–08h) vs afternoon (15–17h) peaks
tot = oh.groupby(["site", "origin"])["count"].sum()
am = oh[oh["hour"].between(6, 8)].groupby(["site", "origin"])["count"].sum()
pm = oh[oh["hour"].between(15, 17)].groupby(["site", "origin"])["count"].sum()
ci = pd.DataFrame({"weekday_records": tot, "am_share": am / tot, "pm_share": pm / tot}).fillna(0).reset_index()
site_ci = pd.DataFrame({"am": oh[oh["hour"].between(6, 8)].groupby("site")["count"].sum(),
                        "pm": oh[oh["hour"].between(15, 17)].groupby("site")["count"].sum()}).div(oh.groupby("site")["count"].sum(), axis=0)

fig, axes = site_axes(1, 3, w=7, h=6)
for ax, s in zip(axes[0], SITE_ORDER):
    p = ci[ci["site"] == s]
    ax.scatter(p["am_share"], p["pm_share"], s=20 + 400 * p["weekday_records"] / p["weekday_records"].max(),
               alpha=0.7, color=SITE_COLORS[s], edgecolor="k")
    for _, r in p.iterrows():
        ax.annotate(r["origin"], (r["am_share"], r["pm_share"]), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.axvline(site_ci.loc[s, "am"], ls="--", color="grey")
    ax.axhline(site_ci.loc[s, "pm"], ls="--", color="grey")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_xlabel("share arriving 06:00–08:59")
    ax.set_ylabel("share arriving 15:00–17:59")
    ax.set_title(f"{s}: commuter signature (dashed = site average)")
show()

ci["peak_share"] = ci["am_share"] + ci["pm_share"]
display(ci.sort_values("peak_share", ascending=False).groupby("site").head(5).sort_values(["site", "peak_share"], ascending=[True, False]))

In [0]:
# Origin mix over time: weekly share of each site's top home areas (100% stacked)
wkly = df.groupBy("site", F.date_trunc("week", "ts").alias("week"), "origin").count().toPandas()
fig, axes = site_axes(3, 1, w=16, h=4.5)
for ax, s in zip(axes[:, 0], SITE_ORDER):
    p = wkly[wkly["site"] == s].copy()
    keep = TOP_ORIGINS[s][:8]
    p["grp"] = np.where(p["origin"].isin(keep), p["origin"], "Other")
    p = p.pivot_table(index="week", columns="grp", values="count", aggfunc="sum", fill_value=0).reindex(columns=keep + ["Other"], fill_value=0)
    p = p.div(p.sum(axis=1), axis=0)
    ax.stackplot(p.index, p.T.values, labels=p.columns, alpha=0.85)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.set_title(f"{s}: weekly home-area mix")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
show()

# Visitor share by month
vm = df.groupBy("site", "month").agg(F.avg((F.col("visitor_type") == "visitor").cast("int")).alias("visitor_share")).toPandas()
fig, ax = plt.subplots(figsize=(13, 4.5))
for s in SITE_ORDER:
    p = vm[vm["site"] == s].sort_values("month")
    ax.plot(p["month"], p["visitor_share"], marker="o", label=s, color=SITE_COLORS[s])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title("Visitor share (outside Metro Vancouver) by month")
ax.legend()
show()

## 6. Dwell time
Minutes attached to the tower. Stats below use **valid** records (≤ 24h) unless stated.
Short dwell = passing through; long dwell = working, waiting or lingering (crowding and security relevance).

In [0]:
dq_site = (dv.groupBy("site", "visitor_type")
             .agg(F.expr("percentile_approx(dwell_time, array(0.1, 0.25, 0.5, 0.75, 0.9, 0.99), 1000)").alias("q"),
                  F.avg("dwell_time").alias("mean"), F.count("*").alias("records"))
             .toPandas())
dq_site[["p10", "p25", "p50", "p75", "p90", "p99"]] = pd.DataFrame(dq_site["q"].tolist(), index=dq_site.index)
display(dq_site.drop(columns="q").sort_values(["site", "visitor_type"]))

In [0]:
# Distribution: histogram (≤ 10h, 5-min bins) and ECDF per site
BIN = 5
h = (dv.filter(F.col("dwell_time") <= 600)
       .withColumn("bin", F.floor(F.col("dwell_time") / BIN) * BIN)
       .groupBy("site", "bin").count().toPandas())
h["share"] = h["count"] / h.groupby("site")["count"].transform("sum")

grid = np.round(np.linspace(0, 1, 101), 2)
ecdf = (dv.groupBy("site").agg(F.expr(f"percentile_approx(dwell_time, array({', '.join(map(str, grid))}), 1000)").alias("q"))
          .toPandas().set_index("site"))

fig, axes = site_axes(1, 3, w=6.5, h=4.5)
for s in SITE_ORDER:
    p = h[h["site"] == s].sort_values("bin")
    for ax in axes[0][:2]:
        ax.step(p["bin"], p["share"], where="post", color=SITE_COLORS[s], label=s)
    axes[0][2].plot(ecdf.loc[s, "q"], grid, color=SITE_COLORS[s], label=s)
axes[0][0].set_title(f"Dwell histogram (≤ 10h, {BIN}-min bins)")
axes[0][1].set_yscale("log")
axes[0][1].set_title("Same, log scale (tail shape)")
axes[0][2].set_xscale("log")
axes[0][2].yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
axes[0][2].set_title("Dwell ECDF (valid records)")
for ax in axes[0]:
    ax.set_xlabel("dwell (min)")
    ax.legend()
show()

In [0]:
# Dwell by home area: median, and share of long stays (≥ 10h)
dor = (dv.groupBy("site", "origin")
         .agg(F.expr("percentile_approx(dwell_time, 0.5)").alias("median_dwell"),
              F.avg((F.col("dwell_time") >= 600).cast("int")).alias("long_share"))
         .toPandas())
fig, axes = plt.subplots(1, 2, figsize=(16, max(6, 0.32 * len(ORIGIN_ORDER))))
for ax, col, fmt, cmap, title in [(axes[0], "median_dwell", ".0f", "YlGnBu", "Median dwell (min)"),
                                  (axes[1], "long_share", ".1%", "Reds", "Share of stays ≥ 10h")]:
    p = dor.pivot(index="origin", columns="site", values=col).reindex(index=ORIGIN_ORDER, columns=SITE_ORDER)
    sns.heatmap(p, annot=True, fmt=fmt, cmap=cmap, ax=ax, cbar=False)
    ax.set_title(title)
    ax.set_ylabel("")
show()

In [0]:
# Dwell segments by site and resident / visitor (all records, invalid shown separately)
sg = df.groupBy("site", "visitor_type", "dwell_segment").count().toPandas()
piv = (sg.pivot_table(index=["site", "visitor_type"], columns="dwell_segment", values="count", fill_value=0)
         .reindex(columns=DWELL_LABELS, fill_value=0)
         .reindex([(s, v) for s in SITE_ORDER for v in ["resident", "visitor"]]))
piv = piv.div(piv.sum(axis=1), axis=0)
piv.index = [f"{s} · {v}" for s, v in piv.index]
ax = piv.iloc[::-1].plot(kind="barh", stacked=True, figsize=(13, 5), colormap="viridis")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_title("Dwell segments by site and resident / visitor")
show()

In [0]:
# Dwell by arrival hour: where do long stays start?
dhr = dv.groupBy("site", "hour").agg(F.expr("percentile_approx(dwell_time, array(0.25, 0.5, 0.75, 0.9), 1000)").alias("q")).toPandas()
dhr[["p25", "p50", "p75", "p90"]] = pd.DataFrame(dhr["q"].tolist(), index=dhr.index)
fig, axes = site_axes(1, 3, w=6, h=4.5, sharey=True)
for ax, s in zip(axes[0], SITE_ORDER):
    p = dhr[dhr["site"] == s].sort_values("hour")
    ax.fill_between(p["hour"], p["p25"], p["p75"], alpha=0.25, color=SITE_COLORS[s], label="IQR")
    ax.plot(p["hour"], p["p50"], marker="o", color=SITE_COLORS[s], label="median")
    ax.plot(p["hour"], p["p90"], ls="--", color="black", label="p90")
    ax.set_xticks(range(0, 24, 2))
    ax.set_xlabel("arrival hour")
    ax.set_title(f"{s}: dwell (min) by arrival hour")
    ax.legend()
show()

## 7. Occupancy (15-min): how many devices are present at once?
A device is present from its arrival slot up to (not including) its departure slot. Occupancy = cumulative arrivals − cumulative departures.
Invalid dwell (> 24h) is excluded. The first `WARMUP_DAYS` are dropped, because devices already present before the data starts aren't in it.

In [0]:
def occupancy(sdf, group_cols):
    """Arrivals, departures and devices present per 15-min slot for each group (group_cols must include 'site')."""
    a = sdf.groupBy(*group_cols, "slot").agg(F.count("*").alias("arrivals"))
    d = sdf.groupBy(*group_cols, F.col("end_slot").alias("slot")).agg(F.count("*").alias("departures"))
    p = a.join(d, [*group_cols, "slot"], "full").fillna(0).toPandas()
    out = []
    for key, g in p.groupby(group_cols):
        key = key if isinstance(key, tuple) else (key,)
        s = key[group_cols.index("site")]
        idx = pd.date_range(SITE_START[s], SITE_END[s], freq=f"{RES_MIN}min")
        g = g.set_index("slot").sort_index()
        pre = (g.loc[g.index < idx[0], "arrivals"] - g.loc[g.index < idx[0], "departures"]).sum()
        g = g[["arrivals", "departures"]].reindex(idx, fill_value=0)
        g["occupancy"] = (g["arrivals"] - g["departures"]).cumsum() + pre
        g = g[g.index >= idx[0].normalize() + pd.Timedelta(days=WARMUP_DAYS)]
        for c, v in zip(group_cols, key):
            g[c] = v
        out.append(g)
    o = pd.concat(out).rename_axis("slot").reset_index()
    o["date"] = o["slot"].dt.normalize()
    o["dow"] = o["slot"].dt.dayofweek
    o["hour"] = o["slot"].dt.hour
    o["sod"] = (o["slot"].dt.hour * 60 + o["slot"].dt.minute) // RES_MIN     # slot of day
    o["holiday"] = o["slot"].dt.strftime("%Y-%m-%d").map(BC_HOLIDAYS)
    return o.sort_values([*group_cols, "slot"]).reset_index(drop=True)

flow = occupancy(dv, ["site"])
seg_occ = occupancy(dv, ["site", "dwell_segment"])
tier_occ = occupancy(dv, ["site", "tier"])

pk = flow.loc[flow.groupby("site")["occupancy"].idxmax(), ["site", "slot", "occupancy"]].set_index("site").loc[SITE_ORDER]
pk["median_when_active"] = flow[flow["occupancy"] > 0].groupby("site")["occupancy"].median()
pk["avg_weekday_daily_peak"] = flow[flow["dow"] < 5].groupby(["site", "date"])["occupancy"].max().groupby("site").mean()
display(pk.reset_index())

In [0]:
# Full timeline (use the range slider to zoom into days)
fig = go.Figure()
for s in SITE_ORDER:
    p = flow[flow["site"] == s]
    fig.add_trace(go.Scattergl(x=p["slot"], y=p["occupancy"], mode="lines", name=s, line=dict(color=SITE_COLORS[s], width=1)))
    for cap in CAPACITY_SCENARIOS.get(s, []):
        fig.add_hline(y=cap, line_dash="dash", line_color=SITE_COLORS[s], annotation_text=f"{s} capacity {cap:,}")
fig.update_layout(title=f"Devices present per {RES_MIN} min", height=500, yaxis_title="devices present")
fig.update_xaxes(rangeslider_visible=True)
fig.show()

for s, caps in CAPACITY_SCENARIOS.items():
    for cap in caps:
        above = flow[(flow["site"] == s) & (flow["occupancy"] > cap)]
        print(f"{s} capacity {cap:,}: exceeded for {len(above) * RES_MIN:,} min over {above['date'].nunique()} days")

In [0]:
# Typical occupancy: weekday × hour (holidays excluded)
fig, axes = plt.subplots(3, 1, figsize=(16, 12))
for ax, s in zip(axes, SITE_ORDER):
    p = flow[(flow["site"] == s) & flow["holiday"].isna()].groupby(["dow", "hour"])["occupancy"].mean().unstack()
    p.index = [DOW[i] for i in p.index]
    sns.heatmap(p, cmap="rocket_r", ax=ax, cbar_kws={"label": "avg devices present"})
    ax.set_title(f"{s}: average devices present, weekday × hour")
show()

# Daily peak over the whole period
dpk = flow.groupby(["site", "date"])["occupancy"].max().reset_index()
fig, ax = plt.subplots(figsize=(16, 5))
for s in SITE_ORDER:
    p = dpk[dpk["site"] == s]
    ax.plot(p["date"], p["occupancy"], label=s, color=SITE_COLORS[s], lw=1.2)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(fmt_k))
ax.set_title("Daily peak devices present")
ax.legend()
show()

In [0]:
# Who makes up the crowd on a typical weekday? (average devices present by slot of day)
def weekday_profile(occ, group):
    o = occ[(occ["dow"] < 5) & occ["holiday"].isna()]
    return o.groupby(["site", group, "sod"])["occupancy"].mean().unstack(group)

xt = np.arange(0, 96, 8)
for occ, group, order, title in [(tier_occ, "tier", TIER_ORDER, "home-area tier"),
                                 (seg_occ, "dwell_segment", DWELL_LABELS[:-1], "dwell segment")]:
    wp = weekday_profile(occ, group)
    fig, axes = site_axes(1, 3, w=6.5, h=4.5)
    for ax, s in zip(axes[0], SITE_ORDER):
        p = wp.loc[s].reindex(columns=order).fillna(0)
        ax.stackplot(p.index, p.T.values, labels=p.columns, alpha=0.85)
        ax.set_xticks(xt)
        ax.set_xticklabels([f"{i * RES_MIN // 60:02d}:00" for i in xt], rotation=45)
        ax.set_title(f"{s}: weekday crowd by {title}")
    axes[0][-1].legend(bbox_to_anchor=(1.01, 1), loc="upper left")
    show()

# Crowd composition at each site's peak slot
at_peak = tier_occ.merge(pk.reset_index()[["site", "slot"]], on=["site", "slot"]).pivot(index="site", columns="tier", values="occupancy")
display(at_peak.reindex(index=SITE_ORDER, columns=TIER_ORDER).reset_index())

## 8. Security signals
Everything here is aggregate and anonymous: counts and distributions, no individual tracking.
Every baseline compares a slot with the **same site, same weekday, same time of day over the previous weeks** (holidays excluded from the baseline).
It only uses past data, so the same logic works for a backtest or a live feed.
* **8a Surges**: arrivals / devices present far above the slot's normal level
* **8b Home-area surges**: one home area suddenly over-represented on a day
* **8c Crowd-mix shift**: the hour's home-area mix departs from that weekday-hour's normal mix (Jensen–Shannon divergence), and which home area drove it
* **8d Rapid build-up**: devices present rising unusually fast over 1 hour
* **8e Long / overnight stays**: devices staying 10h+, and presence between 00:00 and 05:00

In [0]:
# 8a. Surges vs same-weekday / same-slot baseline
hol_mask = flow["holiday"].notna()
flow["arrivals_base"], flow["arrivals_z"] = trailing_z(flow, "arrivals", ["site", "dow", "sod"], exclude=hol_mask)
flow["occupancy_base"], flow["occupancy_z"] = trailing_z(flow, "occupancy", ["site", "dow", "sod"], exclude=hol_mask)
flow["surge"] = (flow["arrivals_z"] > Z_THRESH) & (flow["arrivals"] >= MIN_ARRIVALS)
flow["crowd_high"] = flow["occupancy_z"] > Z_THRESH
flow["lull"] = (flow["arrivals_z"] < -Z_THRESH) & (flow["arrivals_base"] >= MIN_ARRIVALS)   # sudden drop: possible disruption

summary = flow.groupby("site")[["surge", "crowd_high", "lull"]].sum().loc[SITE_ORDER]
summary["slots_scored"] = flow.groupby("site")["arrivals_z"].count()
display(summary.reset_index())

ad = (flow.groupby(["site", "date"])
          .agg(arrivals=("arrivals", "sum"), peak_occupancy=("occupancy", "max"), surge_slots=("surge", "sum"),
               crowd_slots=("crowd_high", "sum"), lull_slots=("lull", "sum"), max_arrivals_z=("arrivals_z", "max"))
          .reset_index())
ad["holiday"] = ad["date"].dt.strftime("%Y-%m-%d").map(BC_HOLIDAYS)
display(ad.sort_values(["surge_slots", "max_arrivals_z"], ascending=False).groupby("site").head(10)
          .sort_values(["site", "surge_slots"], ascending=[True, False]))

fig, axes = site_axes(3, 1, w=16, h=3.2, sharex=True)
for ax, s in zip(axes[:, 0], SITE_ORDER):
    p = ad[ad["site"] == s]
    ax.bar(p["date"], p["surge_slots"], color=SITE_COLORS[s], label="surge slots")
    ax.bar(p["date"], -p["lull_slots"], color="grey", label="lull slots (−)")
    ax.set_title(f"8a. {s}: anomalous {RES_MIN}-min slots per day")
    ax.legend(loc="upper left")
show()

In [0]:
# 8a zoom: the strongest anomaly day at each site, actual vs baseline
fig, axes = site_axes(3, 1, w=16, h=3.8)
for ax, s in zip(axes[:, 0], SITE_ORDER):
    d = ad[ad["site"] == s].sort_values(["surge_slots", "max_arrivals_z"], ascending=False).iloc[0]["date"]
    p = flow[(flow["site"] == s) & (flow["date"] == d)]
    ax.plot(p["slot"], p["arrivals"], color=SITE_COLORS[s], label="arrivals")
    ax.plot(p["slot"], p["arrivals_base"], ls="--", color="black", label="baseline (same weekday / slot)")
    ax.scatter(p.loc[p["surge"], "slot"], p.loc[p["surge"], "arrivals"], color="red", zorder=3, label="surge")
    ax.set_title(f"8a. {s}: {d:%a %Y-%m-%d}" + (f" ({BC_HOLIDAYS[d.strftime('%Y-%m-%d')]})" if d.strftime("%Y-%m-%d") in BC_HOLIDAYS else ""))
    ax.legend()
show()

In [0]:
# 8b. Home-area surges: daily records per site × origin vs the same weekday's trailing baseline
od = df.groupBy("site", "origin", "date").count().toPandas()
od["date"] = pd.to_datetime(od["date"])
od = (od.set_index(["site", "origin", "date"])["count"].unstack("date", fill_value=0)
        .stack().rename("count").reset_index().sort_values(["site", "origin", "date"]).reset_index(drop=True))
od = od[od.apply(lambda r: SITE_START[r["site"]].normalize() <= r["date"] <= SITE_END[r["site"]], axis=1)].reset_index(drop=True)
od["dow"] = od["date"].dt.dayofweek
od["base"], od["z"] = trailing_z(od, "count", ["site", "origin", "dow"], exclude=od["date"].dt.strftime("%Y-%m-%d").isin(BC_HOLIDAYS))
od["surge"] = (od["z"] > Z_THRESH) & (od["count"] >= MIN_ARRIVALS)

osurge = od.groupby(["origin", "site"])["surge"].sum().unstack().reindex(index=ORIGIN_ORDER, columns=SITE_ORDER).fillna(0)
plt.figure(figsize=(8, max(6, 0.32 * len(osurge))))
sns.heatmap(osurge, cmap="Reds", annot=True, fmt=".0f", cbar_kws={"label": "surge days"})
plt.title(f"8b. Home-area surge days (z > {Z_THRESH})")
show()
od["holiday"] = od["date"].dt.strftime("%Y-%m-%d").map(BC_HOLIDAYS)
display(od[od["surge"]].sort_values("z", ascending=False).head(25))

In [0]:
# 8c. Crowd-mix shift: each hour's home-area mix vs the normal mix for that site / weekday / hour
hm = df.groupBy("site", "date", "hour", "origin").count().toPandas()
hm["hour_ts"] = pd.to_datetime(hm["date"]) + pd.to_timedelta(hm["hour"], unit="h")
frames = []
for s in SITE_ORDER:
    P = hm[hm["site"] == s].pivot_table(index="hour_ts", columns="origin", values="count", aggfunc="sum", fill_value=0)
    tot = P.sum(axis=1)
    p = P.div(tot, axis=0)
    Q = P.groupby([P.index.dayofweek, P.index.hour]).transform("sum")
    q = Q.div(Q.sum(axis=1), axis=0)
    m = 0.5 * (p + q)
    with np.errstate(divide="ignore", invalid="ignore"):
        jsd = 0.5 * (p * np.log(p / m)).fillna(0).sum(axis=1) + 0.5 * (q * np.log(q / m)).fillna(0).sum(axis=1)
    diff = p - q
    frames.append(pd.DataFrame({"site": s, "hour_ts": P.index, "arrivals": tot.values,
                                "mix_jsd": jsd.where(tot >= MIN_MIX).values,
                                "driver": diff.idxmax(axis=1).values, "driver_excess": diff.max(axis=1).values}))
mix = pd.concat(frames, ignore_index=True)
mix["date"] = mix["hour_ts"].dt.normalize()
thr = mix.groupby("site")["mix_jsd"].quantile(0.99)

fig, axes = site_axes(3, 1, w=16, h=3.2, sharex=True)
for ax, s in zip(axes[:, 0], SITE_ORDER):
    p = mix[mix["site"] == s].groupby("date")["mix_jsd"].max()
    ax.plot(p.index, p.values, color=SITE_COLORS[s])
    ax.axhline(thr[s], ls="--", color="purple", label="p99 of hours")
    ax.set_title(f"8c. {s}: daily max home-area-mix divergence")
    ax.legend(loc="upper left")
show()
top_mix = mix.dropna(subset=["mix_jsd"]).sort_values("mix_jsd", ascending=False).groupby("site").head(8)
display(top_mix.sort_values(["site", "mix_jsd"], ascending=[True, False])[["site", "hour_ts", "arrivals", "mix_jsd", "driver", "driver_excess"]])

In [0]:
# 8d. Rapid build-up: change in devices present over the last hour vs its own baseline
k = max(1, round(60 / RES_MIN))
flow["occ_change_1h"] = flow.groupby("site")["occupancy"].diff(k)
flow["occ_change_base"], flow["occ_change_z"] = trailing_z(flow, "occ_change_1h", ["site", "dow", "sod"], exclude=hol_mask)
flow["rapid_buildup"] = (flow["occ_change_z"] > Z_THRESH) & (flow["occ_change_1h"] > 0)
print(flow.groupby("site")["rapid_buildup"].sum().loc[SITE_ORDER].to_string())
display(flow[flow["rapid_buildup"]].sort_values("occ_change_z", ascending=False).groupby("site").head(8)
          [["site", "slot", "arrivals", "departures", "occupancy", "occ_change_1h", "occ_change_base", "occ_change_z", "holiday"]])

In [0]:
# 8e. Long / overnight stays
long_occ = (seg_occ[seg_occ["dwell_segment"].isin(DWELL_LABELS[2:4])]
              .groupby(["site", "slot", "hour", "date"])["occupancy"].sum().reset_index())
night = long_occ[long_occ["hour"] < 5].assign(month=lambda x: x["slot"].dt.strftime("%Y-%m"))
nm = night.groupby(["site", "month"])["occupancy"].mean().unstack("site").reindex(columns=SITE_ORDER)

fig, axes = site_axes(1, 2, w=9, h=5)
nm.plot(ax=axes[0][0], marker="o", color=[SITE_COLORS[s] for s in SITE_ORDER])
axes[0][0].set_title("8e. Avg devices present 00:00–05:00 that stay ≥ 3h, by month")
axes[0][0].tick_params(axis="x", rotation=60)
lp = long_occ.groupby(["site", "hour"])["occupancy"].mean().unstack("site").reindex(columns=SITE_ORDER)
lp.plot(ax=axes[0][1], marker="o", color=[SITE_COLORS[s] for s in SITE_ORDER])
axes[0][1].set_title("8e. Avg long-stay (≥ 3h) devices present by hour")
axes[0][1].set_xticks(range(0, 24, 2))
show()

# Share of each home area's records that arrive 00:00–04:59
ns = (df.groupBy("site", "origin").agg(F.avg((F.col("hour") < 5).cast("int")).alias("night_share"))
        .toPandas().pivot(index="origin", columns="site", values="night_share").reindex(index=ORIGIN_ORDER, columns=SITE_ORDER))
plt.figure(figsize=(8, max(6, 0.32 * len(ns))))
sns.heatmap(ns, cmap="Blues", annot=True, fmt=".1%", cbar_kws={"label": "share arriving 00:00–04:59"})
plt.title("8e. Night arrivals by home area")
show()

## 9. Relationships

In [0]:
# Does crowding change behaviour? Hourly arrivals / devices present vs median dwell of that hour's arrivals
hd = (dv.groupBy("site", "date", "hour")
        .agg(F.count("*").alias("arrivals"), F.expr("percentile_approx(dwell_time, 0.5)").alias("median_dwell"))
        .toPandas())
hd["date"] = pd.to_datetime(hd["date"])
hd = hd.merge(flow.groupby(["site", "date", "hour"])["occupancy"].mean().reset_index(), on=["site", "date", "hour"], how="inner")
hd = hd[hd["arrivals"] >= 30]

fig, axes = site_axes(2, 3, w=6, h=4.5)
for j, s in enumerate(SITE_ORDER):
    p = hd[hd["site"] == s]
    for i, xcol in enumerate(["arrivals", "occupancy"]):
        ax = axes[i][j]
        ax.scatter(p[xcol], p["median_dwell"], s=4, alpha=0.3, color=SITE_COLORS[s])
        rho = p[[xcol, "median_dwell"]].corr(method="spearman").iloc[0, 1]
        ax.set_title(f"{s}: {xcol} vs median dwell (ρ = {rho:.2f})")
        ax.set_xlabel("arrivals per hour" if xcol == "arrivals" else "avg devices present")
        ax.set_ylabel("median dwell of arrivals (min)")
show()

In [0]:
# Cross-site: do the sites move together? (daily volume relative to each site's same-weekday normal)
rel = daily.pivot(index="date", columns="site", values="vs_normal")[SITE_ORDER]
print("Correlation of daily deviations from normal:")
display(rel.corr().round(2).reset_index())
wp_corr = prof[prof["day_type"] == "weekday"].pivot(index="hour", columns="site", values="share")[SITE_ORDER].corr()
print("Correlation of weekday hourly profiles:")
display(wp_corr.round(2).reset_index())

# Where does each home area's activity happen? (origins seen at all three sites)
# Raw shares are dominated by site size (the biggest site wins every row), so also show lift:
# lift = home area's share at a site ÷ that site's share of all records. 1 = typical, >1 = over-represented.
op = org_vol.pivot(index="origin", columns="site", values="count").reindex(columns=SITE_ORDER).dropna()
site_share = op.sum() / op.values.sum()
op_share = op.div(op.sum(axis=1), axis=0)
lift = (op_share / site_share).sort_values(SITE_ORDER[0])
op_share = op_share.loc[lift.index]
print("Each site's share of all records (the size effect):", site_share.round(3).to_dict())

fig, axes = plt.subplots(1, 2, figsize=(16, max(6, 0.32 * len(op))))
sns.heatmap(op_share, cmap="Blues", annot=True, fmt=".0%", ax=axes[0], cbar=False)
axes[0].set_title("Raw share of each home area's records (driven by site size)")
sns.heatmap(lift, cmap="vlag", center=1, annot=True, fmt=".2f", ax=axes[1], cbar_kws={"label": "lift (1 = typical)"})
axes[1].set_title("Lift: over / under-representation vs site size")
for ax in axes:
    ax.set_ylabel("")
show()

In [0]:
# Home areas with similar weekday rhythms, across sites (candidate groups for targeted service / messaging)
prof_so = oh.groupby(["site", "origin", "hour"])["count"].sum().unstack(fill_value=0)
prof_so = prof_so[prof_so.sum(axis=1) >= 20_000]
prof_so = prof_so.div(prof_so.sum(axis=1), axis=0)
prof_so.index = [f"{s} · {o}" for s, o in prof_so.index]
if len(prof_so) >= 3:
    g = sns.clustermap(prof_so.T.corr(), cmap="vlag", center=0, figsize=(15, 15), xticklabels=True, yticklabels=True)
    g.ax_heatmap.tick_params(labelsize=7)
    g.fig.suptitle("Correlation of weekday arrival profiles (site · home area, ≥ 20K weekday records)", y=1.02)
    plt.show()

## 10. Catchment map
The data has no origin coordinates, so the lookup below uses **approximate neighbourhood/municipality centroids** (OpenStreetMap Nominatim).
Check the `matched_as` column, and fix any wrong match in `GEOCODE_OVERRIDES`.

In [0]:
import requests, time

GEOCODE_OVERRIDES = {}   # e.g. {"Downtown": (49.2820, -123.1180)}: use if a lookup lands in the wrong place
VIEWBOX = "-123.30,49.40,-122.45,49.00"   # Metro Vancouver (left, top, right, bottom); searches stay inside it

if "GEO_CACHE" not in globals():
    GEO_CACHE = {}

def geocode(name):
    if name in GEOCODE_OVERRIDES:
        return (*GEOCODE_OVERRIDES[name], "manual override")
    if name in GEO_CACHE:
        return GEO_CACHE[name]
    result = (None, None, None)
    for q in [f"{name}, Vancouver, BC, Canada", f"{name}, BC, Canada"]:
        r = requests.get("https://nominatim.openstreetmap.org/search",
                         params={"q": q, "format": "json", "limit": 1, "countrycodes": "ca",
                                 "viewbox": VIEWBOX, "bounded": 1},
                         headers={"User-Agent": "rogers-databricks-hackathon-eda"}, timeout=10)
        time.sleep(1.1)   # Nominatim usage policy: max 1 request/second
        if r.ok and r.json():
            h = r.json()[0]
            result = (float(h["lat"]), float(h["lon"]), h["display_name"])
            break
    GEO_CACHE[name] = result
    return result

site_xy = (df.groupBy("site").agg(F.first("location_name").alias("location_name"),
                                  F.first("latitude").alias("site_lat"), F.first("longitude").alias("site_lon"))
             .toPandas().set_index("site").loc[SITE_ORDER])

local_origins = sorted(org_vol.loc[org_vol["tier"] == "Metro Vancouver", "origin"].unique())
geo = pd.DataFrame([(o, *geocode(o)) for o in local_origins], columns=["origin", "lat", "lon", "matched_as"])
display(geo)
unmatched = geo.loc[geo["lat"].isna(), "origin"].tolist()
if unmatched:
    print("No location found for:", unmatched, "-> add them to GEOCODE_OVERRIDES")

cm = org_vol.merge(geo, on="origin", how="left").merge(site_xy[["site_lat", "site_lon"]], left_on="site", right_index=True)
la1, lo1, la2, lo2 = map(np.radians, [cm["lat"].astype(float), cm["lon"].astype(float), cm["site_lat"], cm["site_lon"]])
cm["km_to_site"] = 6371 * 2 * np.arcsin(np.sqrt(np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2))

In [0]:
# Map: flows from each Metro Vancouver home area to each site (line width = volume)
cmv = cm.dropna(subset=["lat"])
wmax = cmv["count"].max()
fig = go.Figure()
for _, r in cmv.iterrows():
    c = SITE_COLORS[r["site"]].lstrip("#")
    rgba = f"rgba({int(c[0:2], 16)},{int(c[2:4], 16)},{int(c[4:6], 16)},0.45)"
    fig.add_trace(go.Scattermap(lat=[r["lat"], r["site_lat"]], lon=[r["lon"], r["site_lon"]], mode="lines",
                                line=dict(width=1 + 12 * r["count"] / wmax, color=rgba),
                                hoverinfo="skip", showlegend=False))
ot = cmv.groupby(["origin", "lat", "lon"])["count"].sum().reset_index()
fig.add_trace(go.Scattermap(lat=ot["lat"], lon=ot["lon"], mode="markers+text", text=ot["origin"], textposition="top center",
                            marker=dict(size=6 + 24 * ot["count"] / ot["count"].max(), color="#4C78A8"),
                            customdata=ot["count"], hovertemplate="%{text}<br>%{customdata:,} records (all sites)<extra></extra>",
                            name="home areas"))
for s, r in site_xy.iterrows():
    fig.add_trace(go.Scattermap(lat=[r["site_lat"]], lon=[r["site_lon"]], mode="markers+text", text=[s],
                                textposition="bottom center", marker=dict(size=20, color=SITE_COLORS[s]), name=s))
fig.update_layout(map_style="open-street-map", map_zoom=9.6, height=750,
                  map_center=dict(lat=cmv["lat"].mean(), lon=cmv["lon"].mean()),
                  title="Where each site's local visitors live (line colour = site, width = volume)")
fig.show()

In [0]:
# How far do local visitors travel to each site?
def wquantile(x, w, q):
    o = np.argsort(x)
    x, w = np.asarray(x)[o], np.asarray(w)[o]
    return x[np.searchsorted(np.cumsum(w) / w.sum(), q)]

rows = []
for s in SITE_ORDER:
    p = cmv[cmv["site"] == s]
    rows.append({"site": s, "mean_km": np.average(p["km_to_site"], weights=p["count"]),
                 "median_km": wquantile(p["km_to_site"], p["count"], 0.5),
                 "p90_km": wquantile(p["km_to_site"], p["count"], 0.9)})
display(pd.DataFrame(rows))

bands = pd.cut(cmv["km_to_site"], [0, 5, 15, 30, np.inf], labels=["< 5 km", "5–15 km", "15–30 km", "30+ km"])
bd = cmv.groupby(["site", bands], observed=False)["count"].sum().unstack().reindex(SITE_ORDER)
bd = bd.div(bd.sum(axis=1), axis=0)
ax = bd.plot(kind="barh", stacked=True, figsize=(12, 3.5), colormap="viridis_r")
ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
ax.set_title("Distance from home area to site (Metro Vancouver visitors)")
show()

## 11. KPI tables
Gold tables that feed the solution (dashboard, alert thresholds, forecasting, Genie):
* `gold_flow_15m`: one row per site × 15-min slot: arrivals, departures, devices present, baselines, z-scores, flags
* `gold_site_day`: one row per site × day: volume, peak occupancy, anomaly counts, holiday
* `gold_site_origin`: one row per site × home area: volume, shares, dwell, commuter signature, surge days, distance
* `gold_mix_hourly`: one row per site × hour: home-area-mix divergence and which home area drove it
* `gold_site_kpis`: one row per site: headline numbers

In [0]:
gold_flow = flow[["site", "slot", "date", "dow", "sod", "holiday", "arrivals", "departures", "occupancy",
                  "arrivals_base", "arrivals_z", "occupancy_base", "occupancy_z", "occ_change_1h", "occ_change_z",
                  "surge", "crowd_high", "lull", "rapid_buildup"]]
save_pdf(gold_flow, "gold_flow_15m")
save_pdf(ad, "gold_site_day")
save_pdf(mix, "gold_mix_hourly")

site_origin = (org_vol.rename(columns={"count": "records", "share": "share_of_site"})
    .merge(dor, on=["site", "origin"], how="left")
    .merge(ci, on=["site", "origin"], how="left")
    .merge(od.groupby(["site", "origin"])["surge"].sum().rename("surge_days").reset_index(), on=["site", "origin"], how="left")
    .merge(cm[["site", "origin", "km_to_site"]], on=["site", "origin"], how="left"))
site_origin["share_of_origin"] = site_origin["records"] / site_origin.groupby("origin")["records"].transform("sum")
all_site_share = org_vol.groupby("site")["count"].sum() / org_vol["count"].sum()
site_origin["lift"] = site_origin["share_of_origin"] / site_origin["site"].map(all_site_share)   # 1 = typical for the site's size
save_pdf(site_origin, "gold_site_origin")

kpis = pd.DataFrame({
    "records": span["rows"], "days": span["n_days"], "records_per_day": span["rows_per_day"],
    "pct_invalid_dwell": dsum["pct_over_24h"],
    "visitor_share": 1 - tiers["Metro Vancouver"],
    "median_dwell": dq_site.groupby("site").apply(lambda g: np.average(g["p50"], weights=g["records"])),
    "peak_occupancy": pk["occupancy"], "peak_slot": pk["slot"],
    "avg_weekday_daily_peak": pk["avg_weekday_daily_peak"],
    "surge_slots": summary["surge"], "lull_slots": summary["lull"],
    "rapid_buildup_slots": flow.groupby("site")["rapid_buildup"].sum(),
    "top_origin": org_vol.groupby("site").first()["origin"],
}).loc[SITE_ORDER].reset_index().rename(columns={"index": "site"})
kpis["median_dwell"] = kpis["median_dwell"].astype(float)   # records-weighted median of resident/visitor medians (approx.)
display(kpis)
save_pdf(kpis, "gold_site_kpis")
print(f"Saved {SILVER_SCHEMA}.silver_visits and {GOLD_SCHEMA}.gold_flow_15m, gold_site_day, gold_mix_hourly, gold_site_origin, gold_site_kpis")